# GridCombat Autoresearch — Colab Edition

**Runtime required:** Gemini 2.5 Flash-Lite API key (preferred) OR GPU — T4 (free tier) for Qwen 2.5 3B

| Cell | Type   | Purpose |
|------|--------|---------|
| 1    | Python | Mount Drive, set path constants, export as env vars |
| 2    | Bash   | Install Node.js and Python packages |
| 3    | Bash   | Configure git identity |
| 4    | Bash   | Repo setup: restore from Drive bundle or initialise a new git repository |
| 5    | Bash   | First run only: copy JS files, make initial commit, and save bundle to Drive |
| 6    | Python | Option A: Load Qwen 2.5 3B Instruct at 4-bit |
| 6B   | Python | Option B: Gemini 2.5 Flash-Lite API backend (Recommended) |
| 7    | Python | Define all orchestrator functions (read before running 8) |
| 8    | Python | Run the experiment loop |

**Resuming after session expiry:** re-run cells 1, 2, 3, 4, (6 OR 6B), 7, 8. Skip cell 5.

## Cell 1 (Python) — Mount Drive and export paths

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/gridcombat'
WORK_DIR    = '/content/gridcombat'
MODEL_CACHE = '/content/model_cache'  # local Colab disk (~80 GB free) -- model re-downloads each session
BUNDLE_PATH = '/content/drive/MyDrive/gridcombat/repo.bundle'

# Export so bash cells can use $DRIVE_ROOT, $WORK_DIR, etc.
os.environ['DRIVE_ROOT']  = DRIVE_ROOT
os.environ['WORK_DIR']    = WORK_DIR
os.environ['MODEL_CACHE'] = MODEL_CACHE
os.environ['BUNDLE_PATH'] = BUNDLE_PATH

os.makedirs(DRIVE_ROOT,  exist_ok=True)
os.makedirs(WORK_DIR,    exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

print(f'Drive root  : {DRIVE_ROOT}')
print(f'Work dir    : {WORK_DIR}')
print(f'Model cache : {MODEL_CACHE}')
print(f'Bundle path : {BUNDLE_PATH}')
print('Drive mounted OK.')

Mounted at /content/drive
Drive root  : /content/drive/MyDrive/gridcombat
Work dir    : /content/gridcombat
Model cache : /content/model_cache
Bundle path : /content/drive/MyDrive/gridcombat/repo.bundle
Drive mounted OK.


## Cell 2 (Bash) — Install Node.js and Python packages

In [2]:
%%bash
echo '--- Node.js ---'
if ! command -v node &> /dev/null; then
    apt-get install -y nodejs 2>&1 | tail -3
fi
node --version

echo '--- Python packages ---'
pip install -q transformers accelerate bitsandbytes

echo 'Done.'

--- Node.js ---
v20.19.0
--- Python packages ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 6.1 MB/s eta 0:00:00
Done.


## Cell 3 (Bash) — Configure git identity

In [3]:
%%bash
git config --global user.email 'autoresearch@colab.local'
git config --global user.name  'Autoresearch Bot'
git config --global init.defaultBranch main
echo 'Git identity:'
git config --global --list | grep user

Git identity:
user.email=autoresearch@colab.local
user.name=Autoresearch Bot


## Cell 4 (Bash) — Repo setup

Restores from Drive bundle if a previous session exists; otherwise initialises a new git repository.
Safe to re-run on session restart.

In [4]:
%%bash
mkdir -p "$WORK_DIR"
cd "$WORK_DIR"

if [ -f "$BUNDLE_PATH" ]; then
    echo 'Bundle found on Drive -- restoring repo...'
    if [ -d .git ]; then
        echo 'Repo already present, fetching latest from bundle.'
        git fetch "$BUNDLE_PATH" 'refs/heads/*:refs/heads/*'
    else
        git clone "$BUNDLE_PATH" .
    fi
    echo
    echo 'Recent history:'
    git log --oneline -5
    echo 'Repo restored.'
else
    if [ -d .git ]; then
        echo 'Repo already initialised.'
    else
        git init
        echo 'Fresh repo initialised.'
    fi
    echo 'Run Cell 5 to add game files (first run only).'
fi

Bundle found on Drive -- restoring repo...

Recent history:
fd46d7d initial: game files
Repo restored.


Cloning into '.'...


## Cell 5 (Bash) — Copy game files, initial commit, and save bundle

**First run only. Skip on resume.**

Upload `ai.js`, `baseline_ai.js`, `game_core.js`, `evaluate.js` via the Colab
file browser (left sidebar, upload icon) so they appear at `/content/`. Then run this cell.

**Fix applied:** After committing the JS files this cell now calls `git bundle create`
to persist the repo to Drive immediately. Without this step the initial commit would be
lost on session expiry because the working directory `/content/gridcombat` is ephemeral.

In [ ]:
%%bash
UPLOAD_DIR='/content'
cd "$WORK_DIR"

ALL_OK=true
for f in ai.js baseline_ai.js game_core.js evaluate.js; do
    if [ -f "$UPLOAD_DIR/$f" ]; then
        cp "$UPLOAD_DIR/$f" "$WORK_DIR/$f"
        echo "Copied: $f"
    elif [ -f "$WORK_DIR/$f" ]; then
        echo "Already present: $f"
    else
        echo "MISSING: $f -- upload it then re-run this cell"
        ALL_OK=false
    fi
done

if [ "$ALL_OK" = true ]; then
    git add ai.js baseline_ai.js game_core.js evaluate.js
    git commit -m 'initial: game files'
    echo
    echo 'Saving bundle to Drive...'
    git bundle create "$BUNDLE_PATH" --all && echo "Bundle saved to $BUNDLE_PATH" || echo "ERROR: bundle save failed"
    echo
    echo 'Initial commit done. Proceed to Cell 6.'
fi

## Cell 6 (Python) — Load Qwen 2.5 3B Instruct

`Qwen/Qwen2.5-3B-Instruct` is a public, ungated model — no HuggingFace account
or token is required.

Downloads approximately 6 GB on first run to local Colab disk (`/content/model_cache`).
Download time is approximately 3-4 minutes on a new session. The model is not cached
to Drive and will re-download at the start of each session. 4-bit NF4 quantization
uses approximately 4-5 GB VRAM on a T4 (16 GB total), leaving sufficient headroom for
inference. The 7B model was found to consume 13.62 GB of the 14.56 GB available,
leaving insufficient memory for the inference buffer. The 3B model is used instead.
`PYTORCH_ALLOC_CONF=expandable_segments:True` is set to reduce memory
fragmentation and `MAX_NEW_TOKENS` is set to 4096, which provides sufficient
generating a complete `ai.js` file.

**Expected warning — safe to ignore:**
The `transformers`/`huggingface_hub` library attempts to read an `HF_TOKEN`
secret on startup regardless of whether the model requires one. Colab intercepts
this and may display a prompt asking to grant access, or print a warning such as
`UserWarning: Error while fetching HF_TOKEN secret value`. This is normal behaviour
from the library itself, not an error in the notebook. The download will proceed
correctly without a token.

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, cache_dir=MODEL_CACHE, trust_remote_code=True
)

print('Loading model at 4-bit NF4...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=MODEL_CACHE,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)
model.eval()

print(f'Device : {next(model.parameters()).device}')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM   : {used:.1f} GB used / {total:.1f} GB total')
print('Model ready.')

## Cell 6B (Python) — Gemini 2.5 Flash-Lite API backend

**Alternative to Cell 6.** Run this instead of Cell 6 when the GPU runtime is unavailable.
Skip Cell 6 entirely if using this cell.

**Key setup (one-time):** In the Colab left sidebar click the key icon (Secrets),
add a secret named `GEMINI_API_KEY`, and paste your key as the value.
Enable notebook access for the secret. The key is stored encrypted by Google
and is never written to the notebook file or cell output.

**Resuming:** re-run cells 1, 2, 3, 4, 6B, 7, 8. Skip cells 5 and 6.

In [11]:
# Cell 6B — Gemini 2.5 Flash-Lite API backend
# Defines call_local() so Cell 7 and Cell 8 require no changes.

import os
os.system('pip install -q google-generativeai')

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

_gemini_model = genai.GenerativeModel(
    model_name='gemini-2.5-flash-lite',
    system_instruction=(
        'You are an expert game AI engineer. '
        'You reason carefully, make one focused change at a time, '
        'and always follow the output format exactly.'
    )
)

def call_local(prompt):
    response = _gemini_model.generate_content(prompt)
    return response.text

print('Gemini 2.5 Flash-Lite ready.')

Gemini 2.5 Flash-Lite ready.


## Cell 7 (Python) — Define orchestrator

**This cell must be run before Cell 8.** It defines all constants, file helpers,
git utilities, the evaluator, the inference function, and the prompt builder that
Cell 8 depends on. The experiment loop does not start until Cell 8 is run.

Note: Uses pure OS file I/O. Absolutely zero stdout capture, `subprocess`, or pipes.

In [14]:
import os, re, time
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
TEMPERATURE      = 0.4
MAX_EXP          = 0      # 0 = run forever; interrupt kernel to stop
EVAL_TIMEOUT_S   = 120
MAX_RETRIES      = 3
MAX_HISTORY_ROWS = 30
MAX_NEW_TOKENS   = 4096

AI_FILE       = 'ai.js'
BASELINE_FILE = 'baseline_ai.js'
RESULTS_FILE  = 'results.tsv'

REQUIRED_STRINGS = ['runAITurn', 'module.exports', 'shouldDefend']


# ── Logging ───────────────────────────────────────────────────────────────────
def log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f'[{ts}] {msg}', flush=True)


# ── File helpers ──────────────────────────────────────────────────────────────
def read(filename):
    with open(os.path.join(WORK_DIR, filename), 'r', encoding='utf-8') as f:
        return f.read()

def write(filename, content):
    with open(os.path.join(WORK_DIR, filename), 'w', encoding='utf-8') as f:
        f.write(content)

def append_result(commit, win_rate, eval_time, status, description):
    wr   = f'{win_rate:.4f}'  if win_rate  is not None else '0.0000'
    et   = f'{eval_time:.1f}' if eval_time is not None else '0.0'
    desc = description.replace('\t', ' ')[:200]
    with open(os.path.join(WORK_DIR, RESULTS_FILE), 'a', encoding='utf-8') as f:
        f.write(f'{commit}\t{wr}\t{et}\t{status}\t{desc}\n')

def recent_results(n=MAX_HISTORY_ROWS):
    try:
        lines = read(RESULTS_FILE).strip().splitlines()
        header = lines[0] if lines else 'commit\twin_rate\teval_time_s\tstatus\tdescription'
        return '\n'.join([header] + lines[1:][-n:])
    except FileNotFoundError:
        return 'commit\twin_rate\teval_time_s\tstatus\tdescription'

def best_win_rate_from_history():
    best = 50.0
    try:
        for line in read(RESULTS_FILE).strip().splitlines()[1:]:
            parts = line.split('\t')
            if len(parts) >= 4 and parts[3].strip().lower() == 'keep':
                try:
                    wr = float(parts[1])
                    if wr > best: best = wr
                except ValueError:
                    pass
    except FileNotFoundError:
        pass
    return best


# ── Git helpers (Pure File I/O for info gathering) ────────────────────────────
def git_short_hash():
    # Reads directly from .git to avoid subprocess and capturing output entirely.
    try:
        head = read('.git/HEAD').strip()
        if head.startswith('ref: '):
            return read('.git/' + head[5:]).strip()[:7]
        return head[:7]
    except Exception:
        return 'unknown'

def git_commit(message):
    safe = message.replace('"', "'").replace('\n', ' ')[:120]
    os.system('git add ai.js')
    rc = os.system(f'git commit -m "{safe}"') >> 8
    return rc == 0

def git_reset_hard():
    rc = os.system('git reset --hard HEAD~1') >> 8
    if rc != 0:
        log('WARNING: git reset --hard failed. Working tree may be dirty.')

def recent_git_log(n=10):
    # Parses the reflog file directly. No execution required.
    try:
        lines = read('.git/logs/HEAD').strip().splitlines()
        log_lines = []
        for line in reversed(lines[-n:]):
            parts = line.split('\t', 1)
            if len(parts) == 2:
                meta = parts[0].split()
                if len(meta) >= 2:
                    log_lines.append(f"{meta[1][:7]} {parts[1]}")
        return '\n'.join(log_lines) or '(no git history yet)'
    except Exception:
        return '(no git history yet)'

def save_bundle():
    rc = os.system(f'git bundle create "{BUNDLE_PATH}" --all') >> 8
    if rc == 0:
        log(f'  [drive] Bundle saved to {BUNDLE_PATH}')
    else:
        log('  [drive] Bundle save failed.')


# ── Evaluator (File-Backed Node Wrapper) ──────────────────────────────────────
def run_evaluator():
    # We dynamically create a JS runner that writes directly to disk from inside JS.
    # This entirely avoids Python stdout capturing, pipes, and shell redirection.
    js_wrapper = (
        "const fs = require('fs');\n"
        "const logFile = 'eval_out.log';\n"
        "fs.writeFileSync(logFile, '');\n"
        "const writeLog = (msg) => fs.appendFileSync(logFile, msg);\n"
        "process.stdout.write = writeLog;\n"
        "process.stderr.write = writeLog;\n"
        "console.log = (...args) => writeLog(args.join(' ') + '\\n');\n"
        "console.error = (...args) => writeLog(args.join(' ') + '\\n');\n"
        "setTimeout(() => {\n"
        f"    writeLog('\\n[eval] TIMEOUT after {EVAL_TIMEOUT_S}s\\n');\n"
        "    process.exit(124);\n"
        f"}}, {EVAL_TIMEOUT_S} * 1000);\n"
        "try {\n"
        "    require('./evaluate.js');\n"
        "} catch(e) {\n"
        "    writeLog('\\n' + (e.stack || String(e)) + '\\n');\n"
        "    process.exit(1);\n"
        "}\n"
    )

    write('runner.js', js_wrapper)

    # Execute without redirection. All output lives securely inside eval_out.log.
    os.system('node runner.js')

    try:
        out = read('eval_out.log')
    except FileNotFoundError:
        out = ''

    wr = re.search(r'^win_rate:\s+([\d.]+)', out, re.MULTILINE)
    et = re.search(r'^eval_time_s:\s+([\d.]+)', out, re.MULTILINE)

    if not wr:
        log('  [eval] No win_rate in output. Tail:\n' + '\n'.join(out.splitlines()[-20:]))
        return None, None

    return float(wr.group(1)), float(et.group(1)) if et else 0.0


# ── Local inference ───────────────────────────────────────────────────────────

if 'tokenizer' in globals():
    def call_local(prompt):
        messages = [
            {
                'role': 'system',
                'content': (
                    'You are an expert game AI engineer. '
                    'You reason carefully, make one focused change at a time, '
                    'and always follow the output format exactly.'
                )
            },
            {'role': 'user', 'content': prompt},
        ]
        text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors='pt').to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
        return tokenizer.decode(new_ids, skip_special_tokens=True)

elif 'call_local' not in globals():
    def call_local(prompt):
        raise RuntimeError("Error: Run Cell 6 (Local GPU) or Cell 6B (Gemini) before running this cell.")


# ── Prompt ────────────────────────────────────────────────────────────────────
GAME_CONSTANTS_SUMMARY = """
## Game constants (read-only, from game_core.js)

Unit stats:  type         hp  move  capture  ranged  range
             infantry     10    3     yes      no      -
             mech         12    2     yes      no      -
             tank         10    2     no       no      -
             heavy        16    2     no       no      -
             artillery     8    2     no       yes    3-4
             rocket         7    2     no       yes    3-5

Damage table (attacker rows, defender cols):
             vs:  inf  tank  mech  heavy  arty  rocket
  infantry         5    2     3     2      4     3
  tank             8    6     5     4      5     6
  mech             6    5     5     3      6     5
  heavy           10    8     9     6      7     8
  artillery        9    8     8     6      5     7
  rocket           6   10     9     8      6     5

Terrain defense multiplier (lower = more damage taken):
  plain:0.85  wood:0.70  mountain:0.40  road:1.00  water:impassable

UNIT_VALUE: infantry:10  mech:30  tank:70  heavy:160  artillery:60  rocket:150

Combat: finalDamage = floor(baseDamage * (attacker.hp/maxHp) * terrainDef * (1-homeBonus))
Melee only: defender counter-attacks if alive.  Ranged: no counter.
Capture: capturer must stand on enemy HQ each turn; 2-10 capture points/turn.
Win: capture all enemy HQs, or eliminate all enemy units.
"""

def build_prompt(ai_js, results_history, experiment_num, best_wr):
    return f"""You are an autonomous AI researcher. Your job is to improve the game AI
in ai.js for a turn-based strategy game by modifying the heuristic decision logic.

## Current experiment: #{experiment_num}
## Best win_rate so far: {best_wr:.4f}  (baseline = 50.0, higher is better)
## Evaluation: 200 games (4 scenarios x 25 x both sides). Noise ~1.5 points.
{GAME_CONSTANTS_SUMMARY}

## Experiment history (results.tsv -- use this to avoid repeating failures):
{results_history}

## Recent git commits (last 10) -- do NOT reproduce any of these changes exactly:
{recent_git_log()}

## Current ai.js -- the ONLY file you may modify:
```javascript
{ai_js}
```

## Your task
Make ONE focused change to improve win_rate. Think about what has and has not
worked in the history above. Do not repeat a change that was already discarded.

Good targets:
- UNIT_THREAT_WEIGHT or UNIT_CAUTION values at the top
- hqPullWeight or approachWeight inside the movement sort
- shouldDefend() threshold logic
- calculateAttackValue() scoring terms
- Attack priority ordering (who fires first)
- New heuristics (retreat when hp < 30%, focus-fire, flanking bonus)
- Removing a heuristic that may be hurting performance

## Output format -- follow this EXACTLY (the parser is strict):
1. Return the complete modified ai.js inside a single ```javascript code block.
2. Do NOT include any text before the opening ```.
3. After the closing ```, write exactly one line starting with the word CHANGE:
   describing what you changed and your reasoning.

Example:
```javascript
<complete file here>
```
CHANGE: Raised hqPullWeight from 4 to 10 for non-capturers because artillery and
tanks were meandering instead of advancing toward the objective.
"""


# ── Response parsing ──────────────────────────────────────────────────────────
def extract_js(text):
    m = re.search(r'```javascript[^\n]*\n(.*?)```', text, re.DOTALL)
    if m: return m.group(1)
    m = re.search(r'```[^\n]*\n(.*?)```', text, re.DOTALL)
    if m: return m.group(1)
    stripped = text.strip()
    if stripped.startswith(("'use strict'", '"use strict"', '//', '/*')):
        return stripped
    return None

def extract_description(text):
    after_code = re.sub(r'```.*?```', '', text, flags=re.DOTALL).strip()
    m = re.search(r'^CHANGE:\s*(.+)', after_code, re.MULTILINE | re.IGNORECASE)
    if m: return m.group(1).strip()[:200]
    for line in after_code.splitlines():
        line = line.strip()
        if line and not line.startswith('`'):
            return line[:200]
    return 'no description provided'

def passes_sanity_check(code):
    for s in REQUIRED_STRINGS:
        if s not in code:
            return False, f'missing required string: {s}'
    if len(code) < 500:    return False, f'too short ({len(code)} chars)'
    if len(code) > 80_000: return False, f'too long ({len(code)} chars)'
    return True, 'ok'

def check_js_syntax(code):
    write('_tmp_check.js', code)
    rc = os.system('node --check _tmp_check.js 2> _syn_err.txt') >> 8
    if rc != 0:
        err = read('_syn_err.txt').strip()[:400]
        line_match = re.search(r':(\d+)$', err, re.MULTILINE)
        line_num = line_match.group(1) if line_match else 'unknown'
        log(f'Syntax error at line {line_num}:\n{err}')
        return False, err
    return True, ''

print('Orchestrator defined. Run Cell 8 to start.')

Orchestrator defined. Run Cell 8 to start.


## Research Architecture — Two Complementary Loops

This notebook implements one of two intended research loops. They operate at
different levels and are designed to reinforce each other.

### Loop 1 — Autoresearch (this notebook)

Operates continuously and without human intervention. The model makes small,
focused changes to `ai.js` — primarily parameter adjustments and simple heuristic
additions — guided solely by win rate as a signal. It has no visibility into what
actually occurs during games. Its strengths are:

- Parametric optimisation of weights such as `hqPullWeight`, `UNIT_THREAT_WEIGHT`,
  and threat penalty multipliers
- Simple heuristic additions within the existing code structure
- Emergent behavioural improvements as a consequence of better-balanced parameters,
  without any explicit targeted fix being written

Its limitation is that it cannot diagnose systemic problems or produce substantial
algorithmic additions reliably, as win rate alone does not supply sufficient
information for that class of reasoning.

### Loop 2 — Directed Research (session-based, human-guided)

Operates through human observation and chat-based analysis. Known gameplay problems
— such as ranged units advancing past the front line, or units failing to coordinate
— are described in natural language and reasoned about with reference to the game
constants, damage tables, and code structure. This produces deliberate, targeted
changes that address specific diagnosed failures rather than searching blindly.

Its output is injected into `ai.js` manually before resuming the autoresearch loop,
which then refines the result further through parameter tuning.

### Division of Labour

| Class of change | Loop 1 | Loop 2 |
|---|---|---|
| Parameter tuning | Yes | No |
| Simple heuristic additions | Occasionally | Yes |
| Emergent behavioural correction | Yes | No |
| Targeted fix for a known problem | No | Yes |
| Substantial algorithmic additions | No | Yes |

### Injecting a Loop 2 change

1. Interrupt Cell 8 if running.
2. Edit `ai.js` in `/content/gridcombat/` directly with the targeted change.
3. Commit it manually: `git add ai.js && git commit -m 'directed: description'`
4. Save the bundle: `git bundle create "$BUNDLE_PATH" --all`
5. Resume Cell 8. The autoresearch loop will continue from the new baseline.

## Cell 8 (Python) — Run the experiment loop

Stop at any time with **Runtime > Interrupt execution**.
Bundle is saved to Drive on every KEEP so progress survives session expiry.
On session restart re-run cells 1, 2, 3, 4, 6, 7, then this cell.

In [ ]:
import os
os.chdir(WORK_DIR)

for f in [AI_FILE, BASELINE_FILE]:
    if not os.path.exists(os.path.join(WORK_DIR, f)):
        raise FileNotFoundError(f'{f} not found in {WORK_DIR}. Run Cell 5 first.')

if not os.path.exists(os.path.join(WORK_DIR, RESULTS_FILE)):
    write(RESULTS_FILE, 'commit\twin_rate\teval_time_s\tstatus\tdescription\n')

if 'genai' in globals() or '_gemini_model' in globals():
    log('Model           : Gemini 2.5 Flash-Lite (API)')
else:
    log('Model           : Qwen2.5 Local (4-bit NF4)')
log(f'Temperature     : {TEMPERATURE}')
log(f'Max experiments : {"inf" if MAX_EXP == 0 else MAX_EXP}')
log(f'Eval timeout    : {EVAL_TIMEOUT_S}s')
log(f'Work dir        : {WORK_DIR}')
log(f'Drive bundle    : {BUNDLE_PATH}')
log('')

best_wr              = best_win_rate_from_history()
experiment_num       = 0
consecutive_failures = 0
log(f'Best win_rate from history: {best_wr:.4f}')

while True:
    experiment_num += 1
    if MAX_EXP > 0 and experiment_num > MAX_EXP:
        log(f'Reached MAX_EXPERIMENTS={MAX_EXP}. Stopping.')
        break

    log('')
    log('=' * 60)
    log(f'Experiment #{experiment_num}  |  Best: {best_wr:.4f}')
    log('=' * 60)

    # ---- Inference ----
    ai_js  = read(AI_FILE)
    prompt = build_prompt(ai_js, recent_results(), experiment_num, best_wr)
    log('Running local inference...')
    try:
        response_text        = call_local(prompt)
        consecutive_failures = 0
    except Exception as e:
        log(f'Inference error: {e}')
        consecutive_failures += 1
        log(f'Consecutive failures: {consecutive_failures}/{MAX_RETRIES}')
        if consecutive_failures >= MAX_RETRIES:
            log('Too many consecutive failures. Stopping.')
            break
        experiment_num -= 1
        time.sleep(2)
        continue

    # ---- Parse ----
    new_code    = extract_js(response_text)
    description = extract_description(response_text)

    if new_code is None:
        log('Could not extract JS. Skipping.')
        log('Preview: ' + response_text[:400].replace('\n', ' '))
        experiment_num -= 1
        continue

    ok, reason = passes_sanity_check(new_code)
    if not ok:
        log(f'Sanity check failed: {reason}. Skipping.')
        experiment_num -= 1
        continue

    syn_ok, syn_err = check_js_syntax(new_code)
    if not syn_ok:
       log(f'Syntax error (node --check):\n{syn_err}')
       experiment_num -= 1
       continue

    log(f'Proposed: {description}')

    # ---- Commit ----
    write(AI_FILE, new_code)
    if not git_commit(description):
        log('git commit failed (nothing changed). Restoring.')
        rc = os.system('git checkout HEAD -- ai.js') >> 8
        log(f'Restored ai.js from HEAD (rc={rc})')
        experiment_num -= 1
        continue

    commit_hash = git_short_hash()
    log(f'Committed {commit_hash}')

    # ---- Evaluate ----
    log('Running evaluator...')
    win_rate, eval_time = run_evaluator()

    if win_rate is None:
        log('CRASH -- evaluator returned no win_rate. Reverting.')
        git_reset_hard()
        append_result(commit_hash, None, None, 'crash', description)
        continue

    delta = win_rate - best_wr
    log(f'win_rate: {win_rate:.4f}  ({delta:+.4f} vs best)  eval_time: {eval_time:.1f}s')

    # ---- Keep or discard ----
    if win_rate > best_wr:
        best_wr = win_rate
        log(f'KEEP -- new best: {best_wr:.4f}')
        append_result(commit_hash, win_rate, eval_time, 'keep', description)
        save_bundle()
    else:
        git_reset_hard()
        log('DISCARD -- reverted to previous best.')
        append_result(commit_hash, win_rate, eval_time, 'discard', description)

[01:47:22] Model           : Gemini 2.5 Flash-Lite (API)
[01:47:22] Temperature     : 0.4
[01:47:22] Max experiments : inf
[01:47:22] Eval timeout    : 120s
[01:47:22] Work dir        : /content/gridcombat
[01:47:22] Drive bundle    : /content/drive/MyDrive/gridcombat/repo.bundle
[01:47:22] 
[01:47:22] Best win_rate from history: 50.0000
[01:47:22] 
[01:47:22] ============================================================
[01:47:22] Experiment #1  |  Best: 50.0000
[01:47:22] ============================================================
[01:47:22] Running local inference...


## Architecture Note — Two-Loop AI Development Strategy

### Why Gemini 2.5 Flash-Lite is the Correct Choice for the Autoresearch Loop

While local models like Qwen 2.5 3B provide a private, zero-cost alternative, Gemini 2.5 Flash-Lite via API offers a significant leap in reliability and reasoning capability without the VRAM constraints of a T4 GPU. In earlier iterations, smaller local models were found to eventually fixate on specific proposals or lose syntactic coherence during long-form code generation. Gemini 2.5 Flash-Lite resolves these issues, maintaining high output quality across hundreds of experiments.

Inference time and reliability are the primary constraints. Gemini 2.5 Flash-Lite provides a fast turnaround and follows complex instructions more faithfully than 3B-class local models. This allows the search process to focus on productive heuristic exploration rather than overcoming model-induced syntax errors.

### The Two-Loop Strategy

The autoresearch loop above operates within a narrow band of possible improvements. It adjusts parameters and simple heuristics — weights, thresholds, priority orderings — guided by the win rate signal. Beneficial emergent effects are possible, but deliberate algorithmic reasoning is best handled by the second loop.

A second, complementary loop is human-guided and driven by observation. Known gameplay problems are described and reasoned about, producing targeted algorithmic additions. These are injected into `ai.js` manually, and the autoresearch loop then fine-tunes the parameters within that expanded capability.

### Research Strategy and Validation Framework

**The practical criterion:**
The sole criterion of success is whether the game becomes harder to defeat for a human player. While win rate against a baseline is a useful signal for the autonomous loop, direct play testing against human opponents is required for final validation. If AI versus AI improvements do not translate to harder human play, the procedure has no practical value for this project.

## Findings, Conclusions, and Future Directions

### What Was Established

This system constitutes a formal instrument for autonomous game AI research. The following was established empirically:

The autoresearch loop functions correctly and reliably when using Gemini 2.5 Flash-Lite. It handles inference, evaluation, and persistence without human intervention. Significant win rate improvements have been achieved, confirming the pipeline produces real results.

Earlier experiments with Qwen 2.5 3B demonstrated that while 3B-class models can achieve initial gains, they eventually hit a capacity ceiling in agentic use. The transition to Gemini 2.5 Flash-Lite successfully bypassed these limitations, allowing for sustained, autonomous exploration of the heuristic search space.

### The Boundary Condition

The autoresearch loop is a tool for capturing improvements through parameter and heuristic search. It is highly effective at optimising existing logic. Discovering entirely novel algorithms remains a task for the human-guided loop, which provides the structural baseline for the autonomous loop to refine.

### Future Directions

1. **Human play testing.** The current best AI must be tested against human opponents to verify that win rate gains translate to increased tactical difficulty.
2. **Advanced Spatial Models.** Implementing more complex spatial awareness heuristics for ranged units, which the autoresearch loop can then parameterise.
3. **Richer Context.** Feeding more detailed game logs into the prompt to provide the model with a better understanding of *why* certain games were lost, potentially narrowing the search toward even more productive changes.

### A Note on the Contribution

This notebook documents the reasoning, constraints, and successes of the autoresearch approach. It provides a reproducible foundation for iterative AI improvement, demonstrating how API-based models can be integrated into an autonomous research harness to overcome local hardware limitations.